In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install biopython langchain-text-splitters sentence-transformers chromadb -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, time, pandas as pd
from Bio import Entrez

Entrez.email = "your_email@example.com"
DRIVE_PATH = "/content/drive/MyDrive/generag/crispr_abstracts.csv"
os.makedirs("/content/drive/MyDrive/generag", exist_ok=True)

def fetch_pubmed_abstracts(query, max_results=50):
    records = []
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    id_list = Entrez.read(handle)["IdList"]; handle.close()
    print(f"Found {len(id_list)} papers")
    time.sleep(0.5)
    for i in range(0, len(id_list), 10):
        try:
            h = Entrez.efetch(db="pubmed", id=",".join(id_list[i:i+10]), rettype="xml", retmode="xml")
            for r in Entrez.read(h)["PubmedArticle"]:
                try:
                    records.append({
                        "pmid": str(r["MedlineCitation"]["PMID"]),
                        "title": str(r["MedlineCitation"]["Article"]["ArticleTitle"]),
                        "abstract": str(r["MedlineCitation"]["Article"].get("Abstract", {}).get("AbstractText", [""])[0])
                    })
                except KeyError: continue
            h.close()
        except Exception as e: print(f"Batch {i//10+1} failed: {e}")
        time.sleep(0.4)
    return pd.DataFrame(records)

if os.path.exists(DRIVE_PATH):
    df = pd.read_csv(DRIVE_PATH)
    print(f"Loaded {len(df)} abstracts from Drive")
else:
    df = fetch_pubmed_abstracts("CRISPR Cas9 gene editing mechanism", max_results=50)
    df.to_csv(DRIVE_PATH, index=False)
    print(f"Fetched and saved {len(df)} abstracts")

Loaded 50 abstracts from Drive


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " "])
chunks = []
for _, row in df.iterrows():
    for i, text in enumerate(splitter.split_text(f"Title: {row['title']}\n\n{row['abstract']}")):
        chunks.append({
            "chunk_id": f"{row['pmid']}_chunk_{i}",
            "pmid": row['pmid'],
            "title": row['title'],
            "text": text
        })
print(f"Total chunks: {len(chunks)}")

Total chunks: 256


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in chunks]
print("Embedding... (1-2 mins)")
embeddings = model.encode(texts, show_progress_bar=True)

client = chromadb.PersistentClient(path="/content/drive/MyDrive/generag/chroma_db")
collection = client.get_or_create_collection("crispr_abstracts", metadata={"hnsw:space": "cosine"})

for i in range(0, len(chunks), 50):
    batch = chunks[i:i+50]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        embeddings=embeddings[i:i+50].tolist(),
        documents=[c["text"] for c in batch],
        metadatas=[{"pmid": c["pmid"], "title": c["title"]} for c in batch]
    )
print(f"Stored {collection.count()} chunks in ChromaDB")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding... (1-2 mins)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Stored 256 chunks in ChromaDB


In [ ]:
!pip install groq chromadb sentence-transformers langchain-text-splitters biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 471.8 kB/s eta 0:00:00


In [ ]:
import os
from sentence_transformers import SentenceTransformer
import chromadb

DRIVE_BASE = "/content/drive/MyDrive/generag"
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=f"{DRIVE_BASE}/chroma_db")
collection = client.get_collection("crispr_abstracts")
print(f"Loaded ChromaDB with {collection.count()} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded ChromaDB with 256 chunks


In [ ]:
def retrieve(query, k=5):
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text": doc,
            "title": meta["title"],
            "pmid": meta["pmid"],
            "similarity": round(1 - dist, 3)  # cosine distance → similarity
        })
    return chunks

# Test retrieval
test = retrieve("How does Cas9 cut DNA?")
for i, c in enumerate(test):
    print(f"\n[{i+1}] similarity={c['similarity']} | {c['title'][:80]}")
    print(c['text'][:200])


[1] similarity=0.588 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applications.
. Overall, CRISPR-Cas9 represents a transformative but carefully regulated platform for advancing biotechnology and medicine.

[2] similarity=0.584 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applications.
Title: CRISPR Cas9 revolutionizing genetic engineering and therapeutic applications.

[3] similarity=0.566 | The <i>ezrin</i> Gene Regulates Early Cardiac Morphogenesis and Contractile Func
. CRISPR/Cas9 gene editing technology was used to generate <i>ezra</i> knockout lines, and the simultaneous knockdown of <i>ezra</i> and <i>ezrb</i> was induced via morpholino oligonucleotides (MOs)

[4] similarity=0.545 | CRISPR-Cas9-Mediated Gene Editing in Hematological Disorders: Advancing Translat
. Collectively, current evidence supports CRISPR-Cas9 as a cornerstone technology in precision hematology, with continued optimization required to enable safe, equitable, and w

In [ ]:
from groq import Groq

GROQ_API_KEY = "API_KEY"  # paste your key here
groq_client = Groq(api_key=GROQ_API_KEY)

def build_prompt(query, chunks):
    context = ""
    for i, c in enumerate(chunks):
        context += f"\n[Source {i+1}] PMID:{c['pmid']} | {c['title'][:60]}\n{c['text']}\n"

    return f"""You are a CRISPR biology expert assistant. Answer the question using ONLY the provided research abstracts.
If the answer is not in the abstracts, say "Not found in retrieved literature."
Always cite which Source number(s) support your answer.

RESEARCH CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

def ask_generag(query, k=5, verbose=False):
    # Step 1: Retrieve
    chunks = retrieve(query, k=k)

    # Step 2: Generate
    prompt = build_prompt(query, chunks)
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,  # low temp = factual, less hallucination
        max_tokens=512
    )
    answer = response.choices[0].message.content

    if verbose:
        print("=== RETRIEVED CHUNKS ===")
        for i, c in enumerate(chunks):
            print(f"[{i+1}] sim={c['similarity']} | {c['title'][:70]}")
        print("\n=== ANSWER ===")

    return {"query": query, "answer": answer, "sources": chunks}

# Test it
result = ask_generag("What is the role of guide RNA in CRISPR Cas9?", verbose=True)
print(result["answer"])

=== RETRIEVED CHUNKS ===
[1] sim=0.733 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
[2] sim=0.722 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
[3] sim=0.721 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
[4] sim=0.67 | Genetic Interruption of PD-1/PD-L1 as an Alternative Means for Immune 
[5] sim=0.664 | A streamlined approach for gene editing in non-obese diabetes (NOD) mi

=== ANSWER ===
The role of guide RNA in CRISPR Cas9 is to direct Cas9 nuclease activity for targeted DNA editing. [Source 3]


In [ ]:
def verify_answer(answer, chunks):
    """Ask LLM to check if the answer is actually supported by retrieved chunks."""
    context = "\n".join([f"[Source {i+1}]: {c['text']}" for i, c in enumerate(chunks)])

    verify_prompt = f"""You are a fact-checker. Given the following source passages and an answer,
determine if the answer is fully supported, partially supported, or unsupported by the sources.
Respond in this exact JSON format:
{{
  "verdict": "supported" | "partial" | "unsupported",
  "confidence": 0.0-1.0,
  "reason": "one sentence explanation"
}}

SOURCES:
{context}

ANSWER TO VERIFY:
{answer}

JSON VERDICT:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": verify_prompt}],
        temperature=0.0,
        max_tokens=150
    )

    import json
    try:
        verdict = json.loads(response.choices[0].message.content.strip())
    except:
        verdict = {"verdict": "unknown", "confidence": 0.0, "reason": "Parse error"}

    return verdict

def ask_and_verify(query, k=5):
    result = ask_generag(query, k=k)
    verdict = verify_answer(result["answer"], result["sources"])

    print(f"QUERY: {query}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nVERIFICATION:")
    print(f"  Verdict    : {verdict['verdict'].upper()}")
    print(f"  Confidence : {verdict['confidence']}")
    print(f"  Reason     : {verdict['reason']}")
    print(f"\nTOP SOURCES:")
    for c in result["sources"][:3]:
        print(f"  PMID:{c['pmid']} | sim={c['similarity']} | {c['title'][:70]}")

    return {**result, "verification": verdict}

# Run the full agentic pipeline
r1 = ask_and_verify("How does CRISPR Cas9 achieve specificity in gene editing?")

QUERY: How does CRISPR Cas9 achieve specificity in gene editing?

ANSWER:
CRISPR-Cas9 achieves specificity in gene editing through guide RNA-directed Cas9 nuclease activity, allowing targeted DNA editing (Source 1). 

Note: The other sources do not provide additional information on how CRISPR-Cas9 achieves specificity in gene editing beyond what is mentioned in Source 1.

VERIFICATION:
  Verdict    : SUPPORTED
  Confidence : 1.0
  Reason     : The answer is directly stated in Source 1, which explicitly describes the mechanism by which CRISPR-Cas9 achieves specificity in gene editing.

TOP SOURCES:
  PMID:42303162 | sim=0.834 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
  PMID:42303162 | sim=0.827 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
  PMID:42363245 | sim=0.827 | A streamlined approach for gene editing in non-obese diabetes (NOD) mi


In [ ]:
import pandas as pd

test_queries = [
    "What is the role of guide RNA in CRISPR Cas9?",
    "What are off-target effects in CRISPR gene editing?",
    "How is CRISPR used for therapeutic applications?",
    "What are the differences between Cas9 and Cas12?",
    "How do researchers improve CRISPR editing efficiency?"
]

results = []
for q in test_queries:
    print(f"\n{'='*60}\nQ: {q}")
    r = ask_and_verify(q)
    results.append({
        "query": q,
        "answer": r["answer"],
        "verdict": r["verification"]["verdict"],
        "confidence": r["verification"]["confidence"],
        "reason": r["verification"]["reason"],
        "top_source_pmid": r["sources"][0]["pmid"],
        "top_source_similarity": r["sources"][0]["similarity"]
    })

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(f"{DRIVE_BASE}/generag_results.csv", index=False)
print(f"\nSaved results to Drive")
results_df[["query", "verdict", "confidence"]]


Q: What is the role of guide RNA in CRISPR Cas9?
QUERY: What is the role of guide RNA in CRISPR Cas9?

ANSWER:
The role of guide RNA in CRISPR-Cas9 is to direct the Cas9 nuclease activity for targeted DNA editing. [Source 3]

VERIFICATION:
  Verdict    : SUPPORTED
  Confidence : 1.0
  Reason     : The answer is directly stated in Source 3, which explicitly describes the role of guide RNA in CRISPR-Cas9 technology.

TOP SOURCES:
  PMID:42303162 | sim=0.733 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
  PMID:42303162 | sim=0.722 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic
  PMID:42303162 | sim=0.721 | CRISPR Cas9 revolutionizing genetic engineering and therapeutic applic

Q: What are off-target effects in CRISPR gene editing?
QUERY: What are off-target effects in CRISPR gene editing?

ANSWER:
Off-target effects in CRISPR gene editing refer to unintended edits or modifications to the genome at locations other than the intended targ

,query,verdict,confidence
0,What is the role of guide RNA in CRISPR Cas9?,supported,1.0
1,What are off-target effects in CRISPR gene edi...,supported,1.0
2,How is CRISPR used for therapeutic applications?,fully supported,1.0
3,What are the differences between Cas9 and Cas12?,unsupported,1.0
4,How do researchers improve CRISPR editing effi...,unsupported,1.0
